# GeoCLIP grouped attention-intervention random search

This Colab notebook evaluates one global intervention configuration over an entire dataset. The 24 CLIP vision layers are split into three contiguous groups (early, middle, late). A candidate `delta = (delta_1, delta_2, delta_3)` applies `a' = a + delta_i` and `b' = b - delta_i` to every layer in group `i`.

Input bounding boxes are produced outside this notebook. Location-gallery embeddings and the unmodified baseline are computed once and reused throughout the search.

In [ ]:
# Pinned to the Transformers-compatible fork supplied for this experiment.
%pip install -q "git+https://github.com/Le-Anh-Duy/geo-clip.git@976a3d494f43c5f43572348571ba38c28c7877b6"

## 1. Configuration

Set `DATASET_JSON` to the manifest described in the companion Markdown document. Relative image paths are resolved from the manifest directory. The default objective minimizes mean top-1 geodesic distance.

In [ ]:
import json
import math
import random
import types
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm

DATASET_JSON = Path('/content/dataset/manifest.json')
OUTPUT_DIR = Path('/content/geoclip_search_output')

D = 3
BASE_A = 0.0
BASE_B = 0.0
DELTA_LOW = -3.0
DELTA_HIGH = 3.0
NUM_RANDOM_TRIALS = 30
BATCH_SIZE = 8
LOCATION_BATCH_SIZE = 4096
TOP_K = 5
SEED = 42
OBJECTIVE = 'mean_distance_km'  # or 'median_distance_km'; lower is better
THRESHOLDS_KM = (1, 25, 200, 750, 2500)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print('Device:', DEVICE)

## 2. Dataset preparation

Each record contains an image path, zero or more `xyxy` pixel-space boxes, and a ground-truth latitude/longitude. Boxes are mapped through CLIP's resize and center crop here. Records whose resulting patch mask is empty are reported and excluded from both baseline and intervention metrics. Remaining images are CLIP-preprocessed once and kept in CPU memory so random-search trials do not repeat image I/O or resizing.

In [ ]:
def build_in_region_mask(boxes, orig_w, orig_h, grid, image_size):
    scale = image_size / min(orig_w, orig_h)
    resized_w, resized_h = orig_w * scale, orig_h * scale
    crop_x0 = (resized_w - image_size) / 2
    crop_y0 = (resized_h - image_size) / 2
    patch_size = image_size / grid
    mask = torch.zeros((grid, grid), dtype=torch.bool)

    for x0, y0, x1, y1 in boxes:
        x0 = x0 * scale - crop_x0
        x1 = x1 * scale - crop_x0
        y0 = y0 * scale - crop_y0
        y1 = y1 * scale - crop_y0
        x0, x1 = max(x0, 0), min(x1, image_size)
        y0, y1 = max(y0, 0), min(y1, image_size)
        if x1 <= x0 or y1 <= y0:
            continue
        col0 = int(x0 // patch_size)
        col1 = min(int((x1 - 1e-6) // patch_size) + 1, grid)
        row0 = int(y0 // patch_size)
        row1 = min(int((y1 - 1e-6) // patch_size) + 1, grid)
        mask[row0:row1, col0:col1] = True

    return mask.flatten()


def validate_boxes(boxes, width, height, record_id):
    if not isinstance(boxes, list):
        raise ValueError(f'{record_id}: boxes must be a list')
    validated = []
    for box in boxes:
        if not isinstance(box, list) or len(box) != 4:
            raise ValueError(f'{record_id}: each box must be [x1, y1, x2, y2]')
        values = [float(value) for value in box]
        if not all(math.isfinite(value) for value in values):
            raise ValueError(f'{record_id}: box coordinates must be finite')
        x1, y1, x2, y2 = values
        if not (0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height):
            raise ValueError(f'{record_id}: box {box} is outside the {width}x{height} image')
        validated.append(values)
    return validated


def load_and_prepare_dataset(manifest_path, processor, grid, image_size):
    manifest_path = Path(manifest_path)
    records = json.loads(manifest_path.read_text(encoding='utf-8'))
    if not isinstance(records, list) or not records:
        raise ValueError('Manifest must be a non-empty JSON array')

    prepared = []
    empty_mask_ids = []
    seen_ids = set()
    for index, record in enumerate(tqdm(records, desc='Preprocessing dataset')):
        record_id = str(record.get('id', index))
        if record_id in seen_ids:
            raise ValueError(f'Duplicate record id: {record_id}')
        seen_ids.add(record_id)

        image_path = Path(record['image'])
        if not image_path.is_absolute():
            image_path = manifest_path.parent / image_path
        if not image_path.is_file():
            raise FileNotFoundError(f'{record_id}: image not found: {image_path}')

        ground_truth = record['ground_truth']
        lat = float(ground_truth['lat'])
        lon = float(ground_truth['lon'])
        if not (math.isfinite(lat) and -90 <= lat <= 90):
            raise ValueError(f'{record_id}: invalid latitude {lat}')
        if not (math.isfinite(lon) and -180 <= lon <= 180):
            raise ValueError(f'{record_id}: invalid longitude {lon}')

        with Image.open(image_path) as image:
            image = image.convert('RGB')
            width, height = image.size
            boxes = validate_boxes(record['boxes'], width, height, record_id)
            mask = build_in_region_mask(boxes, width, height, grid, image_size)
            if not mask.any():
                empty_mask_ids.append(record_id)
                continue
            pixel_values = processor(images=image, return_tensors='pt')['pixel_values'][0].cpu()

        prepared.append({
            'id': record_id,
            'image': str(image_path),
            'lat': lat,
            'lon': lon,
            'pixel_values': pixel_values,
            'mask': mask,
        })

    if not prepared:
        raise ValueError('No records have a non-empty CLIP patch mask')
    memory_mb = sum(item['pixel_values'].numel() * item['pixel_values'].element_size() for item in prepared) / 2**20
    print(f'Prepared {len(prepared)} images ({memory_mb:.1f} MiB cached tensors); skipped {len(empty_mask_ids)} empty masks')
    return prepared, empty_mask_ids

## 3. Batched attention intervention

This is the same pre-softmax key-bias intervention used by the local sandbox, extended so every image in a batch can carry its own region mask. The CLS key remains neutral.

In [ ]:
@dataclass
class InterventionState:
    layer_ab: dict = field(default_factory=dict)
    in_region_mask: torch.Tensor | None = None

    def key_bias_for_layer(self, layer_idx, num_positions, batch_size):
        a, b = self.layer_ab.get(layer_idx, (0.0, 0.0))
        if (a == 0.0 and b == 0.0) or self.in_region_mask is None:
            return None
        mask = self.in_region_mask
        if mask.ndim == 1:
            mask = mask.unsqueeze(0)
        expected = (batch_size, num_positions - 1)
        if tuple(mask.shape) != expected:
            raise ValueError(f'Expected region mask shape {expected}, got {tuple(mask.shape)}')
        bias = torch.full((batch_size, num_positions), b, dtype=torch.float32, device=mask.device)
        bias[:, 0] = 0.0
        bias[:, 1:].masked_fill_(mask, a)
        return bias[:, None, None, :]


def intervened_attention_forward(state, layer_idx):
    def forward(self, hidden_states, attention_mask=None, **kwargs):
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.head_dim)
        queries = self.q_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        keys = self.k_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        values = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        attention = torch.matmul(queries, keys.transpose(-1, -2)) * self.scale
        if attention_mask is not None:
            attention = attention + attention_mask
        key_bias = state.key_bias_for_layer(layer_idx, attention.shape[-1], attention.shape[0])
        if key_bias is not None:
            attention = attention + key_bias.to(attention)
        attention = F.softmax(attention, dim=-1, dtype=torch.float32).to(queries.dtype)
        output = torch.matmul(attention, values)
        output = output.transpose(1, 2).contiguous().reshape(*input_shape, -1)
        return self.out_proj(output), attention
    return forward


def patch_vision_tower(clip_model):
    state = InterventionState()
    for layer_idx, layer in enumerate(clip_model.vision_model.encoder.layers):
        layer.self_attn.forward = types.MethodType(intervened_attention_forward(state, layer_idx), layer.self_attn)
    return state


def make_group_layer_configs(deltas, num_layers):
    deltas = tuple(float(delta) for delta in deltas)
    if len(deltas) != D:
        raise ValueError(f'Expected {D} deltas, got {len(deltas)}')
    return {
        layer_idx: (BASE_A + deltas[min(layer_idx * D // num_layers, D - 1)],
                    BASE_B - deltas[min(layer_idx * D // num_layers, D - 1)])
        for layer_idx in range(num_layers)
    }

In [ ]:
# Small checks for the non-trivial mask and grouping logic.
assert [min(i * 3 // 24, 2) for i in (0, 7, 8, 15, 16, 23)] == [0, 0, 1, 1, 2, 2]
test_mask = build_in_region_mask([[0, 0, 14, 14]], 224, 224, 16, 224)
assert test_mask.sum().item() == 1 and test_mask[0]
assert not build_in_region_mask([], 224, 224, 16, 224).any()
test_state = InterventionState({0: (2.0, -1.0)}, torch.tensor([[True, False]]))
test_bias = test_state.key_bias_for_layer(0, 3, 1)
torch.testing.assert_close(test_bias.flatten(), torch.tensor([0.0, 2.0, -1.0]))
print('Intervention self-checks passed')

## 4. Load GeoCLIP and cache the location gallery

The location encoder runs once over the fixed GPS gallery in bounded CPU batches. Its normalized outputs are then moved to the accelerator and reused by baseline inference and every random-search trial.

In [ ]:
from geoclip import GeoCLIP

model = GeoCLIP(from_pretrained=True).eval()
gallery_gps = model.gps_gallery.detach().cpu()
location_chunks = []
with torch.inference_mode():
    for start in tqdm(range(0, len(gallery_gps), LOCATION_BATCH_SIZE), desc='Encoding GPS gallery'):
        gps_batch = gallery_gps[start:start + LOCATION_BATCH_SIZE]
        location_chunks.append(F.normalize(model.location_encoder(gps_batch), dim=1).cpu())
location_features = torch.cat(location_chunks).to(DEVICE)
del location_chunks

model = model.to(DEVICE).eval()
clip_model = model.image_encoder.CLIP
intervention_state = patch_vision_tower(clip_model)
num_layers = len(clip_model.vision_model.encoder.layers)
image_size = int(clip_model.vision_model.config.image_size)
grid = image_size // int(clip_model.vision_model.config.patch_size)
assert num_layers == 24 and D == 3, f'Expected 24 layers split into 3 groups, got {num_layers} and D={D}'
print(f'Model ready: {num_layers} layers, {grid}x{grid} patches, {len(gallery_gps):,} cached locations')

In [ ]:
prepared_dataset, empty_mask_ids = load_and_prepare_dataset(
    DATASET_JSON,
    model.image_encoder.image_processor,
    grid,
    image_size,
)
(OUTPUT_DIR / 'empty_patch_mask_ids.json').write_text(json.dumps(empty_mask_ids, indent=2), encoding='utf-8')

## 5. Dataset inference and metrics

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(math.radians, (lat1, lon1, lat2, lon2))
    dlat, dlon = lat2 - lat1, lon2 - lon1
    value = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 6371.0088 * 2 * math.asin(math.sqrt(min(1.0, value)))


def summarize_predictions(frame):
    distances = frame['distance_km'].to_numpy()
    metrics = {
        'mean_distance_km': float(distances.mean()),
        'median_distance_km': float(np.median(distances)),
    }
    metrics.update({f'acc_{threshold}km': float((distances <= threshold).mean()) for threshold in THRESHOLDS_KM})
    return metrics


@torch.inference_mode()
def predict_dataset(prepared, layer_configs=None):
    rows = []
    intervened = layer_configs is not None
    intervention_state.layer_ab = layer_configs or {}
    try:
        for start in range(0, len(prepared), BATCH_SIZE):
            batch = prepared[start:start + BATCH_SIZE]
            pixels = torch.stack([item['pixel_values'] for item in batch]).to(DEVICE)
            intervention_state.in_region_mask = (
                torch.stack([item['mask'] for item in batch]).to(DEVICE) if intervened else None
            )
            image_features = F.normalize(model.image_encoder(pixels), dim=1)
            logits = model.logit_scale.exp() * (image_features @ location_features.T)
            top_values, top_indices = logits.topk(TOP_K, dim=1)
            top_probabilities = (top_values - torch.logsumexp(logits, dim=1, keepdim=True)).exp().cpu()
            top_coordinates = gallery_gps[top_indices.cpu()]

            for item, coordinates, probabilities in zip(batch, top_coordinates, top_probabilities):
                pred_lat, pred_lon = map(float, coordinates[0])
                rows.append({
                    'id': item['id'],
                    'image': item['image'],
                    'gt_lat': item['lat'],
                    'gt_lon': item['lon'],
                    'pred_lat': pred_lat,
                    'pred_lon': pred_lon,
                    'distance_km': haversine_km(pred_lat, pred_lon, item['lat'], item['lon']),
                    'topk_coordinates': json.dumps(coordinates.tolist()),
                    'topk_probabilities': json.dumps(probabilities.tolist()),
                })
    finally:
        intervention_state.layer_ab = {}
        intervention_state.in_region_mask = None
    return pd.DataFrame(rows)


assert OBJECTIVE in {'mean_distance_km', 'median_distance_km'}
assert haversine_km(0, 0, 0, 0) == 0
baseline_predictions = predict_dataset(prepared_dataset)
baseline_metrics = summarize_predictions(baseline_predictions)
baseline_predictions.to_csv(OUTPUT_DIR / 'baseline_per_image.csv', index=False)
pd.Series(baseline_metrics, name='baseline')

## 6. Random search over three group deltas

The zero-delta configuration is evaluated first. With the default `BASE_A = BASE_B = 0`, it reuses the already-computed baseline instead of running the vision encoder again. Every subsequent candidate is scored over the full dataset. Results are checkpointed after every trial.

In [ ]:
rng = np.random.default_rng(SEED)
candidates = [np.zeros(D)]
candidates.extend(rng.uniform(DELTA_LOW, DELTA_HIGH, size=(NUM_RANDOM_TRIALS, D)))

search_rows = []
best_score = math.inf
best_deltas = None
best_predictions = None
best_metrics = None

for trial, deltas in enumerate(tqdm(candidates, desc='Random search')):
    if trial == 0 and BASE_A == 0.0 and BASE_B == 0.0:
        predictions = baseline_predictions.copy()
    else:
        configs = make_group_layer_configs(deltas, num_layers)
        predictions = predict_dataset(prepared_dataset, configs)

    metrics = summarize_predictions(predictions)
    row = {'trial': trial, **{f'delta_{i + 1}': float(value) for i, value in enumerate(deltas)}, **metrics}
    row['mean_improvement_vs_baseline_km'] = baseline_metrics['mean_distance_km'] - metrics['mean_distance_km']
    search_rows.append(row)

    score = metrics[OBJECTIVE]
    if score < best_score:
        best_score = score
        best_deltas = np.asarray(deltas, dtype=float)
        best_predictions = predictions.copy()
        best_metrics = metrics
        best_predictions.to_csv(OUTPUT_DIR / 'best_per_image.csv', index=False)
        groups = [[layer for layer in range(num_layers) if min(layer * D // num_layers, D - 1) == group] for group in range(D)]
        (OUTPUT_DIR / 'best_config.json').write_text(json.dumps({
            'base_a': BASE_A,
            'base_b': BASE_B,
            'deltas': best_deltas.tolist(),
            'layer_groups': groups,
            'objective': OBJECTIVE,
            'score': best_score,
            'metrics': best_metrics,
            'seed': SEED,
        }, indent=2), encoding='utf-8')

    pd.DataFrame(search_rows).sort_values(OBJECTIVE).to_csv(OUTPUT_DIR / 'random_search_results.csv', index=False)

search_results = pd.DataFrame(search_rows).sort_values(OBJECTIVE).reset_index(drop=True)
search_results.head(10)

## 7. Report

In [ ]:
print('Best deltas:', best_deltas.tolist())
display(pd.DataFrame([baseline_metrics, best_metrics], index=['baseline', 'best']))

trial_order = pd.DataFrame(search_rows).sort_values('trial')
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(trial_order['trial'], trial_order[OBJECTIVE].cummin())
axes[0].set(title='Best score so far', xlabel='Trial', ylabel=OBJECTIVE)

x = np.arange(len(THRESHOLDS_KM))
axes[1].bar(x - 0.18, [baseline_metrics[f'acc_{t}km'] for t in THRESHOLDS_KM], 0.36, label='Baseline')
axes[1].bar(x + 0.18, [best_metrics[f'acc_{t}km'] for t in THRESHOLDS_KM], 0.36, label='Best')
axes[1].set(title='Threshold accuracy', xticks=x, xticklabels=[f'{t} km' for t in THRESHOLDS_KM], ylim=(0, 1))
axes[1].legend()

axes[2].hist(baseline_predictions['distance_km'], bins=30, alpha=0.6, label='Baseline')
axes[2].hist(best_predictions['distance_km'], bins=30, alpha=0.6, label='Best')
axes[2].set(title='Top-1 distance distribution', xlabel='Distance (km)')
axes[2].legend()
plt.tight_layout()
plt.show()
print('Outputs:', OUTPUT_DIR)

## 8. Paper artefacts

Every row of the two paper tables is evaluated here from one configuration
list, so the reported numbers cannot drift apart from each other. The
`random_control` row reuses each image's real region mask rolled to a random
location on the patch grid: identical shape and identical patch count, different
content, which separates the effect of the region from the effect of perturbing
attention at all.

Run this after section 6 for the full table. It also runs without the search,
in which case the searched and per-group rows are skipped.

In [ ]:
PAPER_UNIFORM_DELTA = 2.0   # magnitude of the fixed suppress / amplify rows

u = PAPER_UNIFORM_DELTA
HAS_SEARCH = 'best_deltas' in globals() and best_deltas is not None

# Area-matched control masks: same shape, same patch count, rolled to a random
# position on the grid. Offset drawn from 1..grid^2-1, so never the identity.
mask_rng = np.random.default_rng(SEED)
for item in prepared_dataset:
    item['mask_real'] = item['mask']
    dy, dx = divmod(int(mask_rng.integers(1, grid * grid)), grid)
    item['mask_random'] = torch.roll(item['mask'].view(grid, grid), (dy, dx), dims=(0, 1)).flatten()

assert all(i['mask_random'].sum() == i['mask_real'].sum() for i in prepared_dataset)
assert any(not torch.equal(i['mask_random'], i['mask_real']) for i in prepared_dataset)


def use_masks(kind):
    for item in prepared_dataset:
        item['mask'] = item[f'mask_{kind}']


def displacement_km(frame):
    """Geodesic distance between the baseline prediction and this one, per image."""
    merged = baseline_predictions[['id', 'pred_lat', 'pred_lon']].merge(
        frame[['id', 'pred_lat', 'pred_lon']], on='id', suffixes=('_base', '_new'))
    values = [haversine_km(row.pred_lat_base, row.pred_lon_base, row.pred_lat_new, row.pred_lon_new)
              for row in merged.itertuples()]
    return pd.Series(values, index=merged['id'].to_numpy())


assert displacement_km(baseline_predictions).max() == 0

PAPER_CONFIGS = [
    # key, (delta_1, delta_2, delta_3) or None for the unmodified model, mask kind
    ('baseline',       None,          'real'),
    ('suppress',       (-u, -u, -u),  'real'),
    ('amplify',        (u, u, u),     'real'),
    ('random_control', (-u, -u, -u),  'random'),
]
if HAS_SEARCH:
    d1, d2, d3 = (float(value) for value in best_deltas)
    PAPER_CONFIGS += [
        ('searched',     (d1, d2, d3),  'real'),
        ('group_early',  (d1, 0.0, 0.0), 'real'),
        ('group_middle', (0.0, d2, 0.0), 'real'),
        ('group_late',   (0.0, 0.0, d3), 'real'),
    ]
else:
    print('best_deltas not found: skipping the searched and per-group rows')

print(f'{len(PAPER_CONFIGS)} configurations, '
      f'{sum(1 for key, _, _ in PAPER_CONFIGS if key not in {"baseline", "searched"})} to evaluate')

In [ ]:
cached = {'baseline': baseline_predictions}
if HAS_SEARCH:
    cached['searched'] = best_predictions

paper_predictions = {}
paper_rows = []
for key, deltas, mask_kind in tqdm(PAPER_CONFIGS, desc='Paper configurations'):
    if key in cached:
        predictions = cached[key]
    else:
        use_masks(mask_kind)
        predictions = predict_dataset(prepared_dataset, make_group_layer_configs(deltas, num_layers))
    paper_predictions[key] = predictions
    predictions.to_csv(OUTPUT_DIR / f'per_image_{key}.csv', index=False)

    metrics = summarize_predictions(predictions)
    displacement = displacement_km(predictions)
    paper_rows.append({
        'config': key,
        'delta_1': '' if deltas is None else round(deltas[0], 4),
        'delta_2': '' if deltas is None else round(deltas[1], 4),
        'delta_3': '' if deltas is None else round(deltas[2], 4),
        'mask': '' if deltas is None else mask_kind,
        **metrics,
        'mean_error_delta_vs_baseline_km': metrics['mean_distance_km'] - baseline_metrics['mean_distance_km'],
        'mean_displacement_km': float(displacement.mean()),
        'median_displacement_km': float(displacement.median()),
    })
use_masks('real')

paper_table = pd.DataFrame(paper_rows)
paper_table.to_csv(OUTPUT_DIR / 'paper_table.csv', index=False)

(OUTPUT_DIR / 'paper_facts.json').write_text(json.dumps({
    'n_evaluated': len(prepared_dataset),
    'n_excluded_empty_mask': len(empty_mask_ids),
    'num_random_trials': NUM_RANDOM_TRIALS,
    'seed': SEED,
    'uniform_delta': PAPER_UNIFORM_DELTA,
    'delta_range': [DELTA_LOW, DELTA_HIGH],
    'best_deltas': best_deltas.tolist() if HAS_SEARCH else None,
    'objective': OBJECTIVE,
    'thresholds_km': list(THRESHOLDS_KM),
    'num_layers': num_layers,
    'grid': grid,
}, indent=2), encoding='utf-8')

display(paper_table)

In [ ]:
QUALITATIVE_CONFIG = 'suppress'
QUALITATIVE_SAMPLES = 4

by_id = {item['id']: item for item in prepared_dataset}
displacement = displacement_km(paper_predictions[QUALITATIVE_CONFIG])
error_base = baseline_predictions.set_index('id')['distance_km']
error_new = paper_predictions[QUALITATIVE_CONFIG].set_index('id')['distance_km']
chosen = displacement.sort_values(ascending=False).index[:QUALITATIVE_SAMPLES]


def center_crop(path, size):
    """The exact geometry CLIP sees: shortest side to `size`, then a centre crop."""
    with Image.open(path) as source:
        image = source.convert('RGB')
        scale = size / min(image.size)
        image = image.resize((round(image.width * scale), round(image.height * scale)), Image.BICUBIC)
        left, top = (image.width - size) // 2, (image.height - size) // 2
        return np.asarray(image.crop((left, top, left + size, top + size)))


fig, axes = plt.subplots(1, len(chosen), figsize=(1.55 * len(chosen), 1.95), dpi=300)
for axis, image_id in zip(np.atleast_1d(axes), chosen):
    item = by_id[image_id]
    axis.imshow(center_crop(item['image'], image_size))
    overlay = np.zeros((grid, grid, 4))
    overlay[..., 0] = 1.0
    overlay[..., 3] = 0.38 * item['mask_real'].view(grid, grid).numpy()
    axis.imshow(overlay, extent=(0, image_size, image_size, 0), interpolation='nearest')
    axis.set_title(f'shifted {displacement[image_id]:,.0f} km', fontsize=7)
    axis.set_xlabel(f'error {error_base[image_id]:,.0f} $\\rightarrow$ {error_new[image_id]:,.0f} km', fontsize=6)
    axis.set_xticks([])
    axis.set_yticks([])
fig.tight_layout(pad=0.25)
fig.savefig(OUTPUT_DIR / 'qualitative.png', dpi=300, bbox_inches='tight')
plt.show()
print('Wrote', OUTPUT_DIR / 'qualitative.png')

In [ ]:
import time

LATENCY_REPEATS = 20
single = prepared_dataset[:1]
suppress_config = make_group_layer_configs((-u, -u, -u), num_layers)


def time_ms(layer_configs):
    for _ in range(3):
        predict_dataset(single, layer_configs)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(LATENCY_REPEATS):
        predict_dataset(single, layer_configs)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    return (time.perf_counter() - start) / LATENCY_REPEATS * 1000


latency = {
    'device': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'cpu',
    'baseline_ms': time_ms(None),
    'intervened_ms': time_ms(suppress_config),
}
latency['request_ms'] = latency['baseline_ms'] + latency['intervened_ms']
(OUTPUT_DIR / 'latency.json').write_text(json.dumps(latency, indent=2), encoding='utf-8')
print(json.dumps(latency, indent=2))